In [1]:
import pandas as pd
import numpy as np
import os
import re

import json

In [13]:
pd.set_option('display.max_row', None)

In [2]:
lexicon = pd.read_excel('./data/emotion_lexicon/VnEmoLex.xlsx')

# drop the english translation column
lexicon.drop(['Unnamed: 0', 'Unnamed: 12'], axis=1, inplace=True)

# rename columns
lexicon.columns = ['word', 'positive', 'negative', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']

In [3]:
lexicon.head()

,word,positive,negative,anger,anticipation,disgust,fear,joy,sadness,surprise,trust
0,Danh nhân,1,0,0,0,0.0,0,1,0.0,1,1.0
1,danh tiếng,1,0,0,1,0.0,0,1,0.0,1,1.0
2,nghệ thuật nhạc kịch,0,0,0,0,0.0,0,1,0.0,1,0.0
3,sự nổi tiếng,1,1,0,1,0.0,0,1,0.0,1,1.0
4,bất tận,0,0,0,0,0.0,1,0,1.0,0,1.0


In [4]:
lexicon.dtypes

word             object
positive          int64
negative          int64
anger             int64
anticipation      int64
disgust         float64
fear              int64
joy               int64
sadness         float64
surprise          int64
trust           float64
dtype: object

In [5]:
# convert numeric columns to integers
for col in lexicon.columns:
    if col != 'word':
        lexicon[col] = lexicon[col].fillna(0).astype(int)

In [6]:
lexicon['disgust'].unique()

array([0, 1])

In [8]:
# convert words to lowercase
lexicon['word'] = lexicon['word'].apply(lambda x: str(x).lower().strip())

# clean up words
lexicon['word'] = lexicon['word'].apply(lambda x: x.replace(" ", "_"))

# remove unnecessary prefixes 
# Define only the grammatical prefixes to remove
prefixes_to_strip = ['bị_', 'sự_']
pattern = '^(' + '|'.join(prefixes_to_strip) + ')'
# Strip the prefixes
lexicon['word'] = lexicon['word'].str.replace(pattern, '', regex=True)

In [9]:
lexicon.head()

,word,positive,negative,anger,anticipation,disgust,fear,joy,sadness,surprise,trust
0,danh_nhân,1,0,0,0,0,0,1,0,1,1
1,danh_tiếng,1,0,0,1,0,0,1,0,1,1
2,nghệ_thuật_nhạc_kịch,0,0,0,0,0,0,1,0,1,0
3,nổi_tiếng,1,1,0,1,0,0,1,0,1,1
4,bất_tận,0,0,0,0,0,1,0,1,0,1


In [22]:
# 1. Identify duplicates and extract unique words
duplicate_mask = lexicon.duplicated(subset=['word'], keep=False)
unique_words = lexicon[duplicate_mask]['word'].unique()

# 2. Create the new DataFrame for labeling
unique_duplicates_to_label = pd.DataFrame(unique_words, columns=['word'])

# 3. Get all columns from lexicon (except 'word') and initialize with 0
# This assumes your lexicon has columns like 'fear', 'anger', 'positive', etc.
for col in lexicon.columns:
    if col != 'word':
        unique_duplicates_to_label[col] = 0

# 4. Remove the duplicates from the original lexicon to get the cleaned version
lexicon_cleaned = lexicon[~duplicate_mask].copy()

# 5. Save to CSV
output_path = './data/emotion_lexicon/unique_words_to_label.csv'
unique_duplicates_to_label.to_csv(output_path, index=False, encoding='utf-8-sig')
lexicon_cleaned.to_csv('./data/emotion_lexicon/emotion_lexicon_cleaned_part1.csv', index=False)

print(f"File saved to: {output_path}")
print(f"Columns added and zeroed: {[c for c in unique_duplicates_to_label.columns if c != 'word']}")

File saved to: ./data/emotion_lexicon/unique_words_to_label.csv
Columns added and zeroed: ['positive', 'negative', 'anger', 'anticipation', 'disgust', 'fear', 'joy', 'sadness', 'surprise', 'trust']


In [23]:
# after manually labeling the duplicated words
# combine them to the original list
df1 = pd.read_csv('./data/emotion_lexicon/emotion_lexicon_cleaned_part1.csv')
df2 = pd.read_csv('./data/emotion_lexicon/unique_words_to_label.csv')

In [24]:
lexicon = pd.concat([df1, df2], ignore_index=True)

lexicon.head()

,word,positive,negative,anger,anticipation,disgust,fear,joy,sadness,surprise,trust
0,danh_nhân,1,0,0,0,0,0,1,0,1,1
1,danh_tiếng,1,0,0,1,0,0,1,0,1,1
2,nghệ_thuật_nhạc_kịch,0,0,0,0,0,0,1,0,1,0
3,bất_tận,0,0,0,0,0,1,0,1,0,1
4,bệnh_dịch,0,1,1,1,1,1,0,1,1,0


In [25]:
# convert to json
custom_lexicon = {}
emotion_cols = [col for col in lexicon.columns if col !='word']
for _, row in lexicon.iterrows():
    
    word = row['word']
    # keep only emotions where score > 0
    emotion_dict = {emotion: int(row[emotion]) 
                    for emotion in emotion_cols 
                    if row[emotion] > 0}
    
    if len(emotion_dict) > 0:
        custom_lexicon[word] = emotion_dict

In [26]:
custom_lexicon

{'danh_nhân': {'positive': 1, 'joy': 1, 'surprise': 1, 'trust': 1},
 'danh_tiếng': {'positive': 1,
  'anticipation': 1,
  'joy': 1,
  'surprise': 1,
  'trust': 1},
 'nghệ_thuật_nhạc_kịch': {'joy': 1, 'surprise': 1},
 'bất_tận': {'fear': 1, 'sadness': 1, 'trust': 1},
 'bệnh_dịch': {'negative': 1,
  'anger': 1,
  'anticipation': 1,
  'disgust': 1,
  'fear': 1,
  'sadness': 1,
  'surprise': 1},
 'bỏ_cuộc': {'negative': 1, 'anticipation': 1, 'fear': 1, 'sadness': 1},
 'bùng_nổ': {'anger': 1, 'fear': 1, 'joy': 1, 'sadness': 1, 'surprise': 1},
 'cường_điệu': {'positive': 1, 'disgust': 1, 'surprise': 1},
 'dơ': {'negative': 1,
  'anger': 1,
  'anticipation': 1,
  'disgust': 1,
  'fear': 1,
  'sadness': 1,
  'surprise': 1},
 'giáo_dục': {'positive': 1, 'anticipation': 1, 'joy': 1, 'trust': 1},
 'lãng_mạng': {'positive': 1,
  'anticipation': 1,
  'fear': 1,
  'joy': 1,
  'sadness': 1,
  'surprise': 1,
  'trust': 1},
 'mơ_mộng': {'positive': 1,
  'anticipation': 1,
  'fear': 1,
  'joy': 1,
  '

In [27]:
# save to json
with open("./data/emotion_lexicon/custom_emotion_lexicon.json", "w", encoding="utf-8") as f:
    json.dump(custom_lexicon, f, ensure_ascii=False, indent=4)